In [3]:
import os
print(os.getcwd())

/Users/kashakjoshi/Desktop/titanic/notebooks


### Final Prediction & Submission

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd

# Loading processed test data
test_processed = pd.read_csv("../data/processed/test_processed.csv")

# Loading original test for PassengerId
test_raw = pd.read_csv("../data/raw/test.csv")

print(test_processed.shape)

(418, 37)


In [3]:
import joblib
from catboost import CatBoostClassifier

# Loading CatBoost
cat_model = CatBoostClassifier()
cat_model.load_model("../models/catboost_final.cbm")

# Loading Logistic
log_model = joblib.load("../models/logistic_final.pkl")

print("Models loaded successfully.")

Models loaded successfully.


In [5]:
train_processed = pd.read_csv("../data/processed/train_processed.csv")

print("Train columns:", len(train_processed.columns) - 1)
print("Test columns:", len(test_processed.columns))

print("Missing in Test:", set(train_processed.columns) - set(test_processed.columns))
print("Extra in Test:", set(test_processed.columns) - set(train_processed.columns))

Train columns: 37
Test columns: 37
Missing in Test: {'Survived'}
Extra in Test: set()


In [8]:
# Loading train_processed to get exact training column order
train_processed = pd.read_csv("../data/processed/train_processed.csv")

# Exacting feature order used during training
feature_cols = [col for col in train_processed.columns if col != "Survived"]

# Forcing exact order
test_processed = test_processed.reindex(columns=feature_cols)

print("Column order fixed.")

Column order fixed.


In [10]:
print(len(log_model.feature_names_in_))
print(log_model.feature_names_in_)

34
['Pclass' 'Age' 'SibSp' 'Parch' 'Fare' 'FamilySize' 'IsAlone' 'Sex_male'
 'Embarked_Q' 'Embarked_S' 'Title_Miss' 'Title_Mr' 'Title_Mrs'
 'Title_Rare' 'FamilyCategory_Large' 'FamilyCategory_Small' 'Deck_B'
 'Deck_C' 'Deck_D' 'Deck_E' 'Deck_G' 'Deck_Unknown' 'AgeGroup_Teen'
 'AgeGroup_Adult' 'AgeGroup_MidAge' 'AgeGroup_Senior' 'FareGroup_MidLow'
 'FareGroup_MidHigh' 'FareGroup_High' 'Class_Sex_1_male'
 'Class_Sex_2_female' 'Class_Sex_2_male' 'Class_Sex_3_female'
 'Class_Sex_3_male']


In [11]:
expected_cols = log_model.feature_names_in_

test_processed = test_processed.reindex(columns=expected_cols)

print("Forced exact feature alignment.")

Forced exact feature alignment.


In [12]:
cat_pred = cat_model.predict_proba(test_processed)[:, 1]
log_pred = log_model.predict_proba(test_processed)[:, 1]

final_pred = 0.75 * cat_pred + 0.25 * log_pred
final_binary = (final_pred > 0.5).astype(int)

print("Predictions generated successfully.")

Predictions generated successfully.


In [13]:
submission = pd.DataFrame({
    "PassengerId": test_raw["PassengerId"],
    "Survived": final_binary
})

submission.to_csv("../submission.csv", index=False)

print("✅ submission.csv created successfully.")

✅ submission.csv created successfully.


In [14]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


In [15]:
submission["Survived"].value_counts()

Survived
0    269
1    149
Name: count, dtype: int64

In [16]:
submission.dtypes

PassengerId    int64
Survived       int64
dtype: object

In [17]:
submission.head(10)

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0


In [19]:
submission.shape

(418, 2)

In [20]:
print("Min prob:", np.min(final_pred))
print("Max prob:", np.max(final_pred))
print("Mean prob:", np.mean(final_pred))

Min prob: 0.015298563865871695
Max prob: 0.9971742803136052
Mean prob: 0.42483458364028287
